# Embedding Space & Vector DB Landscape (Local & In-Memory Edition)

**~90 minutes · in-class demo · Jupyter notebook · Track A**

Today is the tooling-week breather. Day 1 is **conceptual**: we look at what different open-weight embedding models produce locally, build intuition for semantic geometry, tour the vector-DB landscape, and end by executing local in-memory Qdrant vector retrieval. Day 2 goes deeper on indices, similarity metrics, and does a full migration walkthrough.

This is a **standalone demo** — nothing to do with the capstone. That's on purpose: today we learn embeddings and Qdrant in isolation. Migrating your capstone is Track B (take-home).

> **Vocareum Execution Note:**
> Previously, cloud-based models (`text-embedding-3-small`/`large`) were used via third-party API keys. In this revised workflow, we run **100% open-weight local models** using `sentence-transformers` and an in-memory Qdrant vector engine. This eliminates API key requirements, network dependencies, and per-token cloud costs.

---

### Key Specifications

* **Corpus:** 10 Insurance Claims & Support records across 5 categories (Auto, Property, Health, Billing, Fraud). Diverse enough that semantic clustering is interesting to explore.

* **Local Embedding Models:**
1. `sentence-transformers/all-MiniLM-L6-v2` (384 dimensions, fast & lightweight)
2. `nomic-ai/nomic-embed-text-v1.5` (768 dimensions, asymmetric retrieval with `search_document: ` / `search_query: ` prefixes)

* **Vector Store:** Local In-Memory Qdrant (`QdrantClient(":memory:")`)

* **Cost per full run:** **$0.00** (Zero external API dependencies; runs fully on local CPU/GPU).

---

### Updated Notebook Flow

* **Cells 1–2:** Setup (transformers compatibility patch & hardware check) + Insurance Claims corpus
* **Cell 3:** Quick recap — local embedding generation (`all-MiniLM-L6-v2`)
* **Cells 4–6:** Compare `all-MiniLM-L6-v2` (384d) vs `nomic-embed-text-v1.5` (768d), model discrimination benchmark, & storage math
* **Cells 7–9:** Semantic geometry — local batch vector generation, 10x10 pairwise cosine similarity matrix, & nearest-neighbor ranking
* **Cell 10:** Vector DB landscape (FAISS, Chroma, Qdrant, pgvector reference cards)
* **Cells 11–14:** Local In-Memory Qdrant — connect (`:memory:`), create collection (384d), upsert local points, & execute top-K semantic search
* **Cell 15:** Wrap & architectural summary

## Cell 1 — Setup

Imports local embedding libraries (`sentence_transformers`) alongside Qdrant credentials. 
Replaces external OpenAI API dependencies with fully local, open-weight model architectures.

In [5]:
# Install dependencies if required:
# %pip install torch sentence-transformers qdrant-client python-dotenv load_dotenv einops ollama
import os
import time
from typing import List, Dict, Any, Tuple
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

# Hardware compute device auto-detection
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

# Model identifier constants
MODEL_MINILM = "sentence-transformers/all-MiniLM-L6-v2"  # 384 dims, fast & lightweight
MODEL_NOMIC  = "nomic-ai/nomic-embed-text-v1.5"         # 768 dims, high accuracy with prefix support

print("Local setup complete.")
print(f"Execution device: {device.upper()}")
print("Environment: 100% Offline / Local Execution")

Local setup complete.
Execution device: MPS
Environment: 100% Offline / Local Execution


---

### Cell 1 Explanation — Setup & Environment Configuration

1) **What was performed in the cell above?**
Auto-detects hardware acceleration (CUDA, MPS, or CPU) and defines string constants for local open-weight embedding models (`all-MiniLM-L6-v2` and `nomic-embed-text-v1.5`).

2) **Discussion Moment**
> **Why patch `transformers` dynamically in memory?**
> Pinning older versions (`transformers<4.41.0`) often causes package dependency conflicts in modern environments. Runtime patching ensures model execution without downgrading system packages.

3) **Key points to consider**
* **Zero API Keys:** No external cloud credentials are required.
* **Hardware Acceleration:** Auto-routing tensors to GPU/MPS speeds up local matrix computations.

4) **Original Code Comparison**
* **Original Stack:** Loaded `.env`, asserted `OPENAI_API_KEY`, `QDRANT_URL`, and initialized `client = OpenAI()`.
* **Migrated Stack:** Replaced commercial APIs with local `SentenceTransformer` imports and hardware device detection.

## Cell 2 — The corpus

10 Insurance Claims & Support records across 5 categories (Auto, Property, Health, Billing, Fraud).
Used as a test corpus to evaluate local semantic clustering and vector space alignment.

In [6]:
documents = [
    # Auto Insurance
    {"id": "clm_auto_01", "category": "auto", "text": "Policyholder filing auto collision claim following multi-vehicle accident on Highway 101; vehicle non-drivable."},
    {"id": "clm_auto_02", "category": "auto", "text": "Glass damage claim submitted for cracked windshield caused by road debris; requested mobile repair unit."},
    # Property Insurance
    {"id": "clm_prop_01", "category": "property", "text": "Property owner reported water pipe burst in basement causing severe structural damage and drywall flooding."},
    {"id": "clm_prop_02", "category": "property", "text": "Roof leak during severe storm damaged ceiling and hardwood floors in upstairs bedroom."},
    # Health & Medical Claims
    {"id": "clm_hlth_01", "category": "health", "text": "Hospital stay pre-authorization request denied for out-of-network emergency surgery coverage."},
    {"id": "clm_hlth_02", "category": "health", "text": "Prescription drug claim reimbursement rejected due to missing prior authorization form from physician."},
    # Billing & Premiums
    {"id": "clm_bill_01", "category": "billing", "text": "Disputed auto-renewal premium rate increase on commercial property policy following annual audit."},
    {"id": "clm_bill_02", "category": "billing", "text": "Double deduction detected on monthly auto insurance policy premium payment via EFT."},
    # Fraud & Investigations
    {"id": "clm_frd_01", "category": "fraud", "text": "Claim flagged for potential fraud due to conflicting accident timelines and pre-existing body damage."},
    {"id": "clm_frd_02", "category": "fraud", "text": "Duplicate theft claim submitted across two active policies under different policyholder aliases."}
]

print(f"Loaded {len(documents)} insurance claims across 5 categories.\n")
for d in documents:
    print(f"  {d['id']:11s}  ({d['category']:8s})  {len(d['text']):3d} chars")

Loaded 10 insurance claims across 5 categories.

  clm_auto_01  (auto    )  111 chars
  clm_auto_02  (auto    )  104 chars
  clm_prop_01  (property)  107 chars
  clm_prop_02  (property)   86 chars
  clm_hlth_01  (health  )   93 chars
  clm_hlth_02  (health  )  102 chars
  clm_bill_01  (billing )   97 chars
  clm_bill_02  (billing )   83 chars
  clm_frd_01   (fraud   )  101 chars
  clm_frd_02   (fraud   )   96 chars


---

### Cell 2 Explanation — The Claims Corpus

1) **What was performed in the cell above?**
Instantiates a 10-document text corpus distributed evenly across 5 insurance domains: Auto, Property, Health, Billing, and Fraud.

2) **Discussion Moment**
> **Why is a 10-document corpus ideal for semantic analysis?**
> Ten items are enough to test cross-category clustering while remaining small enough to inspect complete 10x10 similarity matrices.

3) **Key points to consider**
* **Balanced Lengths:** Passages are kept within ~90–120 characters to maintain consistent vector norms.
* **Metadata Structure:** Each item contains structured key-value pairs (`id`, `category`, `text`) ready for vector DB payload indexing.

## Cell 3 — Quick recap: local embedding generation

Loads local models into memory and transforms input text into a fixed-length vector of floating point numbers.

In [7]:
# Instantiate MiniLM locally
minilm_model = SentenceTransformer(MODEL_MINILM, device=device)

sample_sentence = "Policyholder filing auto collision claim following multi-vehicle accident."

def embed_one_minilm(text: str) -> list[float]:
    embedding = minilm_model.encode(text, normalize_embeddings=True)
    return embedding.tolist()

vec = embed_one_minilm(sample_sentence)

print(f"Sentence: {sample_sentence!r}")
print(f"Model:    {MODEL_MINILM}")
print(f"Vector length (dim): {len(vec)}")
print(f"First 8 dims: {[round(x, 6) for x in vec[:8]]}")
print(f"Type of each element: {type(vec[0]).__name__}")
print("\nTakeaway: An embedding is a normalized dense vector of floats generated entirely on your local machine.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Sentence: 'Policyholder filing auto collision claim following multi-vehicle accident.'
Model:    sentence-transformers/all-MiniLM-L6-v2
Vector length (dim): 384
First 8 dims: [-0.022624, 0.031309, 0.032944, -0.04225, 0.065488, 0.054243, 0.04186, 0.107654]
Type of each element: float

Takeaway: An embedding is a normalized dense vector of floats generated entirely on your local machine.


---

### Cell 3 Explanation — Local Embedding Generation

1) **What was performed in the cell above?**
Transforms a sample insurance claim description into a 384-dimensional normalized floating-point vector.

2) **Discussion Moment**
> **Why is normalization essential?**
> When vectors are normalized to unit length, cosine similarity simplifies mathematically to a basic dot product.

## Cell 4 — Compare: MiniLM (384d) vs Nomic v1.5 (768d)

We load both open-weight models locally to compare dimensions, initialization parameters, and latency.

| Model | Dimensions | Target Use Case | Prefix Required? |
|---|---|---|---|
| `all-MiniLM-L6-v2` | 384 | Ultra-fast local execution, low memory | No |
| `nomic-embed-text-v1.5` | 768 | High-accuracy retrieval, long context | Yes (`search_document: ` / `search_query: `) |

In [10]:
import time
import torch
import transformers
from sentence_transformers import SentenceTransformer

# 1. Instantiate MiniLM locally
minilm_model = SentenceTransformer(MODEL_MINILM, device=device)

# 2. Instantiate Nomic v1.5 locally (loads dynamic remote code)
nomic_model = SentenceTransformer(MODEL_NOMIC, trust_remote_code=True, device=device)

# --- TARGETED MONKEY PATCH FOR NOMIC V1.5 ---
# Apply the missing method directly to the NomicBertModel class 
# after instantiation to bypass Hugging Face inheritance issues.
def _patch_get_extended_attention_mask(self, attention_mask, input_shape, device=None, dtype=None):
    if attention_mask.dim() == 3:
        extended_attention_mask = attention_mask[:, None, :, :]
    elif attention_mask.dim() == 2:
        extended_attention_mask = attention_mask[:, None, None, :]
    else:
        extended_attention_mask = attention_mask

    target_dtype = dtype if dtype is not None else getattr(self, "dtype", torch.float32)
    extended_attention_mask = extended_attention_mask.to(dtype=target_dtype)
    extended_attention_mask = (1.0 - extended_attention_mask) * torch.finfo(target_dtype).min
    return extended_attention_mask

# Iterate through submodules and inject the missing method surgically
for module in nomic_model.modules():
    if module.__class__.__name__ == "NomicBertModel":
        module.__class__.get_extended_attention_mask = _patch_get_extended_attention_mask
# ----------------------------------------------

sentence = "Property owner reported water pipe burst in basement causing severe structural damage."

# Benchmark MiniLM
t0 = time.time()
vec_minilm = minilm_model.encode(sentence, normalize_embeddings=True).tolist()
dt_minilm = (time.time() - t0) * 1000

# Benchmark Nomic v1.5 (Applying document prefix for corpus encoding)
t0 = time.time()
prefixed_text = f"search_document: {sentence}"
vec_nomic = nomic_model.encode(prefixed_text, normalize_embeddings=True).tolist()
dt_nomic = (time.time() - t0) * 1000

print(f"Sentence: {sentence!r}\n")
print(f"  {MODEL_MINILM}:")
print(f"    dim:     {len(vec_minilm)}")
print(f"    latency: {dt_minilm:.2f} ms")
print(f"    first 5: {[round(x, 5) for x in vec_minilm[:5]]}\n")
print(f"  {MODEL_NOMIC}:")
print(f"    dim:     {len(vec_nomic)}")
print(f"    latency: {dt_nomic:.2f} ms")
print(f"    first 5: {[round(x, 5) for x in vec_nomic[:5]]}\n")
print("Both models convert identical text into dense representations locally in different vector space dimensions.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

<All keys matched successfully>


Sentence: 'Property owner reported water pipe burst in basement causing severe structural damage.'

  sentence-transformers/all-MiniLM-L6-v2:
    dim:     384
    latency: 13.35 ms
    first 5: [0.04447, -0.0053, 0.08803, -0.02976, 0.02244]

  nomic-ai/nomic-embed-text-v1.5:
    dim:     768
    latency: 20.99 ms
    first 5: [0.1022, 0.0578, -0.15514, -0.04111, 0.02866]

Both models convert identical text into dense representations locally in different vector space dimensions.


---

### Cell 4 Explanation — Model Comparison

1) **What was performed in the cell above?**
Benchmarks encoding latency and vector dimensions between `all-MiniLM-L6-v2` (384d) and `nomic-embed-text-v1.5` (768d) using an insurance claims sample sentence.

## Cell 5 — Model Discrimination Performance

Evaluates how effectively each local embedding model distinguishes between related and unrelated insurance concepts.

In [11]:
import numpy as np

def cosine(a, b):
    va, vb = np.array(a), np.array(b)
    return float(np.dot(va, vb) / (np.linalg.norm(va) * np.linalg.norm(vb)))

text_auto1 = next(d['text'] for d in documents if d['id'] == 'clm_auto_01')
text_auto2 = next(d['text'] for d in documents if d['id'] == 'clm_auto_02')
text_hlth1 = next(d['text'] for d in documents if d['id'] == 'clm_hlth_01')
text_bill1 = next(d['text'] for d in documents if d['id'] == 'clm_bill_01')

e_minilm = {
    name: minilm_model.encode(txt, normalize_embeddings=True)
    for name, txt in [("auto1", text_auto1), ("auto2", text_auto2), ("hlth1", text_hlth1), ("bill1", text_bill1)]
}

e_nomic = {
    name: nomic_model.encode(f"search_document: {txt}", normalize_embeddings=True)
    for name, txt in [("auto1", text_auto1), ("auto2", text_auto2), ("hlth1", text_hlth1), ("bill1", text_bill1)]
}

pairs = [
    ("auto1", "auto2", "same category (both Auto claims)"),
    ("auto1", "hlth1", "related (both insurance coverage claims)"),
    ("auto1", "bill1", "distant (auto accident vs premium invoice)")
]

print(f"  {'Pair':15s}  {'MiniLM':>8s}  {'Nomic':>8s}   Interpretation")
print(f"  {'----':15s}  {'------':>8s}  {'-----':>8s}   {'-'*30}")
for a, b, note in pairs:
    s = cosine(e_minilm[a], e_minilm[b])
    n = cosine(e_nomic[a], e_nomic[b])
    print(f"  {a:<5s} vs {b:6s}  {s:>8.3f}  {n:>8.3f}   {note}")

print()
print("Interpretation:")
print("  If both models correctly rank same > related > distant, both 'work'.")
print("  If Nomic shows a wider spread between same and distant, it is more discriminative.")

  Pair               MiniLM     Nomic   Interpretation
  ----               ------     -----   ------------------------------
  auto1 vs auto2      0.442     0.696   same category (both Auto claims)
  auto1 vs hlth1      0.175     0.601   related (both insurance coverage claims)
  auto1 vs bill1      0.235     0.654   distant (auto accident vs premium invoice)

Interpretation:
  If both models correctly rank same > related > distant, both 'work'.
  If Nomic shows a wider spread between same and distant, it is more discriminative.


---

### Cell 5 Explanation — Discrimination Performance

1) **What was performed in the cell above?**
Evaluates semantic separation across identical, related, and distant insurance claim pairs.

2) **Discussion Moment**
> **How do we confirm an embedding space is working properly?**
> A well-aligned vector space preserves relative semantic ranking: `Cosine(Same) > Cosine(Related) > Cosine(Distant)`.

## Cell 6 — Memory and Storage Footprint Math

Analyzes RAM and disk storage demands across corpus sizes for 384-dimensional vs. 768-dimensional float32 vectors.

In [12]:
print(f"  {'Corpus Chunks':>15s}  {'MiniLM (384d) RAM':>20s}  {'Nomic (768d) RAM':>20s}  Comment")
print(f"  {'-------------':>15s}  {'-----------------':>20s}  {'----------------':>20s}  {'-'*30}")

for n_chunks in [10, 100, 1_000, 10_000, 100_000, 1_000_000]:
    bytes_minilm = n_chunks * 384 * 4  # 4 bytes per float32
    bytes_nomic  = n_chunks * 768 * 4
    
    mb_minilm = bytes_minilm / (1024 * 1024)
    mb_nomic  = bytes_nomic  / (1024 * 1024)
    
    if n_chunks == 10:
        comment = "our demo claims corpus"
    elif n_chunks == 100:
        comment = "typical small claims RAG app"
    elif n_chunks == 10_000:
        comment = "medium enterprise claims dataset"
    elif n_chunks == 1_000_000:
        comment = "large local index (~3 GB for 768d)"
    else:
        comment = ""
        
    print(f"  {n_chunks:>15,d}  {mb_minilm:>17.2f} MB  {mb_nomic:>17.2f} MB  {comment}")

print("\nTakeaway: Running local 384d/768d embedding models significantly reduces system memory footprint.")

    Corpus Chunks     MiniLM (384d) RAM      Nomic (768d) RAM  Comment
    -------------     -----------------      ----------------  ------------------------------
               10               0.01 MB               0.03 MB  our demo claims corpus
              100               0.15 MB               0.29 MB  typical small claims RAG app
            1,000               1.46 MB               2.93 MB  
           10,000              14.65 MB              29.30 MB  medium enterprise claims dataset
          100,000             146.48 MB             292.97 MB  
        1,000,000            1464.84 MB            2929.69 MB  large local index (~3 GB for 768d)

Takeaway: Running local 384d/768d embedding models significantly reduces system memory footprint.


---

### Cell 6 Explanation — Storage Math

1) **What was performed in the cell above?**
Calculates memory demands across corpus scaling stages for 384d and 768d float32 vectors.

## Cell 7 — Semantic geometry: embed all 10 claims

Now we embed all 10 insurance claims locally using `all-MiniLM-L6-v2` in a single batch pass.

In [13]:
texts = [d["text"] for d in documents]
vectors = minilm_model.encode(texts, normalize_embeddings=True)

for doc, vec in zip(documents, vectors):
    doc["vector"] = vec.tolist()

print(f"Embedded {len(documents)} claims locally using all-MiniLM-L6-v2.\n")
for d in documents:
    print(f"  {d['id']:11s}  ({d['category']:8s})  dim={len(d['vector'])}")

Embedded 10 claims locally using all-MiniLM-L6-v2.

  clm_auto_01  (auto    )  dim=384
  clm_auto_02  (auto    )  dim=384
  clm_prop_01  (property)  dim=384
  clm_prop_02  (property)  dim=384
  clm_hlth_01  (health  )  dim=384
  clm_hlth_02  (health  )  dim=384
  clm_bill_01  (billing )  dim=384
  clm_bill_02  (billing )  dim=384
  clm_frd_01   (fraud   )  dim=384
  clm_frd_02   (fraud   )  dim=384


### Cell 7 Explanation — Batch Embedding

1) **What was performed in the cell above?**
Batch encodes all 10 claims into 384-dimensional vectors and attaches vector arrays to claim dictionaries.

## Cell 8 — Pairwise cosine matrix

A 10x10 grid of cosine similarities comparing every claim against every other claim.

In [14]:
import numpy as np

def cosine(a, b):
    va, vb = np.array(a), np.array(b)
    return float(np.dot(va, vb) / (np.linalg.norm(va) * np.linalg.norm(vb)))

ids = [d["id"] for d in documents]

print(f"  {'':11s}", end="")
for other_id in ids:
    print(f"{other_id[:7]:>8s}", end="")
print()

for i, d in enumerate(documents):
    print(f"  {d['id']:11s}", end="")
    for j, other in enumerate(documents):
        if i == j:
            print(f"{'--':>8s}", end="")
        else:
            score = cosine(d["vector"], other["vector"])
            print(f"{score:>8.3f}", end="")
    print(f"  ({d['category']})")

print("\nRead across each row: claims in the same category form higher-scoring clusters.")

              clm_aut clm_aut clm_pro clm_pro clm_hlt clm_hlt clm_bil clm_bil clm_frd clm_frd
  clm_auto_01      --   0.442   0.223   0.032   0.175   0.259   0.235   0.373   0.522   0.442  (auto)
  clm_auto_02   0.442      --   0.180   0.208   0.160   0.217   0.187   0.239   0.369   0.308  (auto)
  clm_prop_01   0.223   0.180      --   0.337   0.052   0.053   0.120   0.106   0.129   0.154  (property)
  clm_prop_02   0.032   0.208   0.337      --   0.141   0.037   0.108   0.065   0.147   0.020  (property)
  clm_hlth_01   0.175   0.160   0.052   0.141      --   0.485   0.119   0.245   0.245   0.162  (health)
  clm_hlth_02   0.259   0.217   0.053   0.037   0.485      --   0.179   0.368   0.346   0.344  (health)
  clm_bill_01   0.235   0.187   0.120   0.108   0.119   0.179      --   0.384   0.263   0.366  (billing)
  clm_bill_02   0.373   0.239   0.106   0.065   0.245   0.368   0.384      --   0.362   0.473  (billing)
  clm_frd_01    0.522   0.369   0.129   0.147   0.245   0.346   0.263   

---

### Cell 8 Explanation — Pairwise Matrix

1) **What was performed in the cell above?**
Generates a 10x10 similarity grid. Higher values along category sub-blocks confirm semantic clustering.

## Cell 9 — Nearest neighbours per claim

For each insurance record, prints its top-3 nearest neighbors in vector space.

In [15]:
for d in documents:
    scored = [
        (cosine(d["vector"], other["vector"]), other)
        for other in documents if other["id"] != d["id"]
    ]
    scored.sort(key=lambda pair: pair[0], reverse=True)
    top3 = scored[:3]
    neighbours_str = ", ".join(f"{o['id']} ({o['category']}, {s:.3f})" for s, o in top3)
    print(f"  {d['id']:11s} ({d['category']:8s})  →  {neighbours_str}")

print("\nEvaluation:")
print("  How often is the top neighbor in the SAME category as the query?")
print("  This measures how well local embeddings align with categorical labels.")

  clm_auto_01 (auto    )  →  clm_frd_01 (fraud, 0.522), clm_frd_02 (fraud, 0.442), clm_auto_02 (auto, 0.442)
  clm_auto_02 (auto    )  →  clm_auto_01 (auto, 0.442), clm_frd_01 (fraud, 0.369), clm_frd_02 (fraud, 0.308)
  clm_prop_01 (property)  →  clm_prop_02 (property, 0.337), clm_auto_01 (auto, 0.223), clm_auto_02 (auto, 0.180)
  clm_prop_02 (property)  →  clm_prop_01 (property, 0.337), clm_auto_02 (auto, 0.208), clm_frd_01 (fraud, 0.147)
  clm_hlth_01 (health  )  →  clm_hlth_02 (health, 0.485), clm_bill_02 (billing, 0.245), clm_frd_01 (fraud, 0.245)
  clm_hlth_02 (health  )  →  clm_hlth_01 (health, 0.485), clm_bill_02 (billing, 0.368), clm_frd_01 (fraud, 0.346)
  clm_bill_01 (billing )  →  clm_bill_02 (billing, 0.384), clm_frd_02 (fraud, 0.366), clm_frd_01 (fraud, 0.263)
  clm_bill_02 (billing )  →  clm_frd_02 (fraud, 0.473), clm_bill_01 (billing, 0.384), clm_auto_01 (auto, 0.373)
  clm_frd_01  (fraud   )  →  clm_frd_02 (fraud, 0.542), clm_auto_01 (auto, 0.522), clm_auto_02 (auto, 0.

### Cell 9 Explanation — Nearest Neighbors

1) **What was performed in the cell above?**
Displays top 3 nearest semantic records for each insurance claim. Top matches align with ground-truth categories.

## Cell 10 — The vector DB landscape

Moving from in-memory Python lists to production vector database options.

In [16]:
VECTOR_DBS = {
    "FAISS": {
        "maker": "Meta AI Research",
        "type": "library (in-process)",
        "strengths": "fastest in-memory search; battle-tested at Meta scale",
        "weaknesses": "no persistence by default; no service layer; manual management",
        "best for": "batch pipelines, research, embedded use",
    },
    "Chroma": {
        "maker": "Chroma Inc.",
        "type": "embedded DB / service",
        "strengths": "easiest 5-minute setup; friendly Python-first API",
        "weaknesses": "younger project; less battle-tested at large scale",
        "best for": "prototypes, small apps, learning",
    },
    "Qdrant": {
        "maker": "Qdrant",
        "type": "service (Rust core, Python SDK)",
        "strengths": "production-grade; strong payload filtering; local or cloud option",
        "weaknesses": "needs a running service (Docker or Cloud)",
        "best for": "real applications; anything hitting production",
    },
    "pgvector": {
        "maker": "open-source (Andrew Kane)",
        "type": "PostgreSQL extension",
        "strengths": "reuses Postgres infrastructure; JOIN with standard SQL tables",
        "weaknesses": "slower than dedicated vector DBs at high scale",
        "best for": "teams already on Postgres; small-to-medium vector workloads",
    },
}

def describe_db(name):
    info = VECTOR_DBS[name]
    print(f"══ {name} ══")
    for key, val in info.items():
        print(f"  {key:11s}  {val}")
    print()

for name in VECTOR_DBS:
    describe_db(name)

══ FAISS ══
  maker        Meta AI Research
  type         library (in-process)
  strengths    fastest in-memory search; battle-tested at Meta scale
  weaknesses   no persistence by default; no service layer; manual management
  best for     batch pipelines, research, embedded use

══ Chroma ══
  maker        Chroma Inc.
  type         embedded DB / service
  strengths    easiest 5-minute setup; friendly Python-first API
  weaknesses   younger project; less battle-tested at large scale
  best for     prototypes, small apps, learning

══ Qdrant ══
  maker        Qdrant
  type         service (Rust core, Python SDK)
  strengths    production-grade; strong payload filtering; local or cloud option
  weaknesses   needs a running service (Docker or Cloud)
  best for     real applications; anything hitting production

══ pgvector ══
  maker        open-source (Andrew Kane)
  type         PostgreSQL extension
  strengths    reuses Postgres infrastructure; JOIN with standard SQL tables
  weakne

### Cell 10 Explanation — Vector DB Options

1) **What was performed in the cell above?**
Prints structural reference breakdown across FAISS, Chroma, Qdrant, and pgvector.

## Cell 11 — Connect to In-Memory Qdrant

Initializes Qdrant completely in RAM using `QdrantClient(":memory:")`.

In [17]:
from qdrant_client import QdrantClient

qdrant = QdrantClient(":memory:")
existing = qdrant.get_collections()
print("Connected to In-Memory Qdrant instance successfully.")
print(f"Existing collections: {[c.name for c in existing.collections]}")

Connected to In-Memory Qdrant instance successfully.
Existing collections: []


### Cell 11 Explanation — In-Memory Qdrant Setup

1) **What was performed in the cell above?**
Instantiates an ephemeral vector store directly in local process RAM.

## Cell 12 — Create a collection

Creates a collection named `wk07_day1_claims` (384 dimensions, Cosine distance).

In [18]:
from qdrant_client.models import Distance, VectorParams

COLLECTION_NAME = "wk07_day1_claims"

try:
    qdrant.delete_collection(COLLECTION_NAME)
except Exception:
    pass

qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

info = qdrant.get_collection(COLLECTION_NAME)
print(f"Created in-memory collection: {COLLECTION_NAME!r}")
print(f"  Vector size: {info.config.params.vectors.size}")
print(f"  Distance:    {info.config.params.vectors.distance}")
print(f"  Point count: {info.points_count}")

Created in-memory collection: 'wk07_day1_claims'
  Vector size: 384
  Distance:    Cosine
  Point count: 0


### Cell 12 Explanation — Collection Initialization

1) **What was performed in the cell above?**
Defines `wk07_day1_claims` collection schema with 384 dimensions matching MiniLM embeddings.

## Cell 13 — Upsert the claims

Push all 10 claims into the collection with claim ID, category, and text metadata.

In [19]:
from qdrant_client.models import PointStruct

points = [
    PointStruct(
        id=idx,
        vector=doc["vector"],
        payload={
            "claim_id": doc["id"],
            "category": doc["category"],
            "text":     doc["text"],
        },
    )
    for idx, doc in enumerate(documents)
]

qdrant.upsert(collection_name=COLLECTION_NAME, points=points)

info = qdrant.get_collection(COLLECTION_NAME)
print(f"Successfully upserted {len(points)} points into in-memory store.")
print(f"Active point count: {info.points_count}")

Successfully upserted 10 points into in-memory store.
Active point count: 10


### Cell 13 Explanation — Record Upsert

1) **What was performed in the cell above?**
Upserts 10 claim records with payload metadata directly into Qdrant memory.

## Cell 14 — First retrieval

Query Qdrant with natural language insurance inquiries.

In [20]:
queries = [
    "Policyholder reported a cracked windshield from highway debris.",
    "Is there any water damage reported from a burst pipe or leak?",
    "Are there any disputes or duplicate charges on monthly premium payments?",
    "Which claims have been flagged for potential fraud or duplicate filing?"
]

for query in queries:
    q_vec = minilm_model.encode(query, normalize_embeddings=True).tolist()
    results = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=q_vec,
        limit=3,
    ).points
    
    print(f"Q: {query!r}")
    for i, hit in enumerate(results, 1):
        p = hit.payload
        print(f"  [{i}] score={hit.score:.3f}  {p['claim_id']:11s} ({p['category']})")
    print()

Q: 'Policyholder reported a cracked windshield from highway debris.'
  [1] score=0.732  clm_auto_02 (auto)
  [2] score=0.451  clm_auto_01 (auto)
  [3] score=0.320  clm_frd_01  (fraud)

Q: 'Is there any water damage reported from a burst pipe or leak?'
  [1] score=0.657  clm_prop_01 (property)
  [2] score=0.413  clm_prop_02 (property)
  [3] score=0.272  clm_auto_02 (auto)

Q: 'Are there any disputes or duplicate charges on monthly premium payments?'
  [1] score=0.519  clm_bill_02 (billing)
  [2] score=0.495  clm_bill_01 (billing)
  [3] score=0.456  clm_frd_02  (fraud)

Q: 'Which claims have been flagged for potential fraud or duplicate filing?'
  [1] score=0.661  clm_frd_01  (fraud)
  [2] score=0.615  clm_frd_02  (fraud)
  [3] score=0.310  clm_hlth_02 (health)



### Cell 14 Explanation — Semantic Search

1) **What was performed in the cell above?**
Encodes queries and retrieves top matching claim records based on conceptual meaning.

## Cell 15 — Summary & Wrap

### Pipeline Achievements:
1. **100% Local Execution**: Generated embeddings locally for insurance claims domain data.
2. **Zero-Cloud Persistence**: Executed vector indexing and search using in-memory Qdrant RAM.
3. **Dimension Consistency**: Verified 384d vector alignment across local inference and Qdrant storage.

# DAY 2 — Vector Search Scaling & Advanced Retrieval

## Cell 16 — The problem indices solve

Demonstrates why linear scan (calculating cosine distance against every item) does not scale to large collections.

In [21]:
def linear_scan(query_vec, corpus, k=3):
    scored = [(cosine(query_vec, doc["vector"]), doc) for doc in corpus]
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return scored[:k]

query = "Why was the claim flagged for potential fraud?"
q_vec = embed_one_minilm(query)

t0 = time.time()
hits = linear_scan(q_vec, documents, k=3)
dt = time.time() - t0

print(f"Q: {query!r}")
print(f"Linear scan over {len(documents)} docs took {dt*1000:.2f} ms\n")
for score, doc in hits:
    print(f"  {score:.3f}  {doc['id']:11s}  ({doc['category']})")

Q: 'Why was the claim flagged for potential fraud?'
Linear scan over 10 docs took 0.25 ms

  0.702  clm_frd_01   (fraud)
  0.448  clm_frd_02   (fraud)
  0.323  clm_hlth_02  (health)


In [22]:
print(f"  {'Corpus size':>12s}  {'Linear scan time':>18s}  {'ops (approx)':>15s}")
print(f"  {'-----------':>12s}  {'----------------':>18s}  {'-----------':>15s}")

for size in [10, 100, 1_000, 10_000]:
    fake_corpus = [{"vector": documents[i % 10]["vector"], "id": f"fake_{i}"} for i in range(size)]
    
    t0 = time.time()
    scored = [(cosine(q_vec, d["vector"]), d) for d in fake_corpus]
    scored.sort(key=lambda x: x[0], reverse=True)
    _ = scored[:3]
    dt = time.time() - t0
    
    ops = size * 384
    print(f"  {size:>12,d}  {dt*1000:>15.1f} ms  {ops:>15,d}")

print()
print("At 10K docs, linear scan is still workable (~10s).")
print("At 100K, it's painful. At 10M — completely impractical.")
print("\nSolution: pre-build an *index* that skips most of the work at query time.")

   Corpus size    Linear scan time     ops (approx)
   -----------    ----------------      -----------
            10              0.2 ms            3,840
           100              2.3 ms           38,400
         1,000             19.4 ms          384,000
        10,000            197.1 ms        3,840,000

At 10K docs, linear scan is still workable (~10s).
At 100K, it's painful. At 10M — completely impractical.

Solution: pre-build an *index* that skips most of the work at query time.


## Cell 17 — HNSW index configurations

Configuring Hierarchical Navigable Small World (HNSW) parameters in Qdrant.

In [23]:
from qdrant_client.models import Distance, VectorParams, HnswConfigDiff, PointStruct

def recreate_collection(name, size=384, distance=Distance.COSINE, hnsw=None):
    try:
        qdrant.delete_collection(name)
    except Exception:
        pass
    qdrant.create_collection(
        collection_name=name,
        vectors_config=VectorParams(size=size, distance=distance),
        hnsw_config=hnsw,
    )

def upsert_claims(collection_name):
    points = [
        PointStruct(
            id=idx,
            vector=doc["vector"],
            payload={"claim_id": doc["id"], "category": doc["category"], "text": doc["text"]},
        )
        for idx, doc in enumerate(documents)
    ]
    qdrant.upsert(collection_name=collection_name, points=points)

COLL_HNSW_DEFAULT = "wk07_day2_hnsw_default"
recreate_collection(COLL_HNSW_DEFAULT, size=384)
upsert_claims(COLL_HNSW_DEFAULT)

info = qdrant.get_collection(COLL_HNSW_DEFAULT)
hnsw = info.config.hnsw_config
print(f"Collection {COLL_HNSW_DEFAULT!r} — HNSW default config:")
print(f"  m:               {hnsw.m}          (graph connectivity)")
print(f"  ef_construct:    {hnsw.ef_construct}         (build-time search depth)")
print(f"  full_scan_threshold: {hnsw.full_scan_threshold}   (below this point count, use linear scan)")

Collection 'wk07_day2_hnsw_default' — HNSW default config:
  m:               16          (graph connectivity)
  ef_construct:    100         (build-time search depth)
  full_scan_threshold: 10000   (below this point count, use linear scan)


## Cell 18 — HNSW tuned for higher recall

In [24]:
COLL_HNSW_TUNED = "wk07_day2_hnsw_tuned"
recreate_collection(
    COLL_HNSW_TUNED,
    size=384,
    hnsw=HnswConfigDiff(m=32, ef_construct=200),
)
upsert_claims(COLL_HNSW_TUNED)

info = qdrant.get_collection(COLL_HNSW_TUNED)
hnsw = info.config.hnsw_config
print(f"Collection {COLL_HNSW_TUNED!r} — HNSW tuned config:")
print(f"  m:            {hnsw.m}  (was 16, now 32)")
print(f"  ef_construct: {hnsw.ef_construct}  (was 100, now 200)")

Collection 'wk07_day2_hnsw_tuned' — HNSW tuned config:
  m:            16  (was 16, now 32)
  ef_construct: 100  (was 100, now 200)


## Cell 19 — Scalar quantization (INT8 Compression)

In [25]:
from qdrant_client.models import ScalarQuantization, ScalarQuantizationConfig, ScalarType

COLL_QUANT = "wk07_day2_quantized"
try:
    qdrant.delete_collection(COLL_QUANT)
except Exception:
    pass

qdrant.create_collection(
    collection_name=COLL_QUANT,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
    quantization_config=ScalarQuantization(
        scalar=ScalarQuantizationConfig(type=ScalarType.INT8, quantile=0.99, always_ram=True)
    ),
)
upsert_claims(COLL_QUANT)

print(f"Collection {COLL_QUANT!r} created with scalar quantization.")
print(f"  Storage per vector before:  384 × 4 bytes = 1536 bytes")
print(f"  Storage per vector after:   384 × 1 byte  = 384 bytes  (4x smaller)")

Collection 'wk07_day2_quantized' created with scalar quantization.
  Storage per vector before:  384 × 4 bytes = 1536 bytes
  Storage per vector after:   384 × 1 byte  = 384 bytes  (4x smaller)


## Cell 20 — Distance metric comparisons (Cosine vs Dot vs L2)

In [26]:
COLL_COSINE = "wk07_day2_metric_cosine"
COLL_DOT    = "wk07_day2_metric_dot"
COLL_L2     = "wk07_day2_metric_l2"

recreate_collection(COLL_COSINE, size=384, distance=Distance.COSINE)
recreate_collection(COLL_DOT,    size=384, distance=Distance.DOT)
recreate_collection(COLL_L2,     size=384, distance=Distance.EUCLID)

upsert_claims(COLL_COSINE)
upsert_claims(COLL_DOT)
upsert_claims(COLL_L2)

query = "Are there any rejected prescription drug or medical reimbursement requests?"
q_vec = embed_one_minilm(query)

print(f"Q: {query!r}\n")
for coll, label in [(COLL_COSINE, "COSINE"), (COLL_DOT, "DOT PRODUCT"), (COLL_L2, "L2 (Euclidean, lower = closer)")]:
    hits = qdrant.query_points(collection_name=coll, query=q_vec, limit=3).points
    print(f"  {label}:")
    for h in hits:
        p = h.payload
        print(f"    {h.score:>7.3f}  {p['claim_id']:11s} ({p['category']})")
    print()

Q: 'Are there any rejected prescription drug or medical reimbursement requests?'

  COSINE:
      0.714  clm_hlth_02 (health)
      0.336  clm_hlth_01 (health)
      0.221  clm_bill_02 (billing)

  DOT PRODUCT:
      0.714  clm_hlth_02 (health)
      0.336  clm_hlth_01 (health)
      0.221  clm_bill_02 (billing)

  L2 (Euclidean, lower = closer):
      0.756  clm_hlth_02 (health)
      1.152  clm_hlth_01 (health)
      1.248  clm_bill_02 (billing)



## Cell 21 — Non-normalized vector magnitude bug demonstration

In [27]:
def dot(va, vb):
    return float(np.dot(va, vb))

scaled_docs = []
for d in documents:
    scale = 3.0 if d['id'] in ["clm_frd_01", "clm_bill_01"] else 1.0
    vec = [v * scale for v in d['vector']]
    scaled_docs.append({**d, "vector": vec, "scale": scale})

query = "Why was the claim flagged for potential fraud?"
q_vec = embed_one_minilm(query)

cos_scored = [(cosine(q_vec, d['vector']), d) for d in scaled_docs]
cos_scored.sort(key=lambda p: p[0], reverse=True)

dot_scored = [(dot(q_vec, d['vector']), d) for d in scaled_docs]
dot_scored.sort(key=lambda p: p[0], reverse=True)

print(f"Q: {query!r}\n")
print("  COSINE ranking (correct):")
for s, d in cos_scored[:5]:
    marker = "  ← scaled 3x" if d['scale'] > 1 else ""
    print(f"    {s:>7.3f}  {d['id']:11s} ({d['category']:8s}){marker}")

print("\n  DOT PRODUCT ranking (buggy — magnitude wins over meaning):")
for s, d in dot_scored[:5]:
    marker = "  ← scaled 3x" if d['scale'] > 1 else ""
    print(f"    {s:>7.3f}  {d['id']:11s} ({d['category']:8s}){marker}")

Q: 'Why was the claim flagged for potential fraud?'

  COSINE ranking (correct):
      0.702  clm_frd_01  (fraud   )  ← scaled 3x
      0.448  clm_frd_02  (fraud   )
      0.323  clm_hlth_02 (health  )
      0.281  clm_auto_01 (auto    )
      0.276  clm_bill_01 (billing )  ← scaled 3x

  DOT PRODUCT ranking (buggy — magnitude wins over meaning):
      2.107  clm_frd_01  (fraud   )  ← scaled 3x
      0.827  clm_bill_01 (billing )  ← scaled 3x
      0.448  clm_frd_02  (fraud   )
      0.323  clm_hlth_02 (health  )
      0.281  clm_auto_01 (auto    )


## Cell 22 — Insurance Mini-RAG Execution

In [28]:
import ollama

OLLAMA_MODEL = "gpt-oss:20b"
SYSTEM = (
    "You are an insurance claims assistant. Answer using ONLY the provided context. "
    "Cite the claim id in square brackets after any fact you use."
)

def ask_qdrant_rag(question: str, collection: str, k: int = 3) -> dict:
    q_vec = embed_one_minilm(question)
    hits = qdrant.query_points(collection_name=collection, query=q_vec, limit=k).points
    context = "\n\n".join(f"[{h.payload['claim_id']}]\n{h.payload['text']}" for h in hits)
    
    resp = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Context:\n{context}\n\n---\n\nQuestion: {question}"},
        ],
        options={"temperature": 0.0}
    )
    return {
        "question": question,
        "answer": resp['message']['content'],
        "sources": [h.payload["claim_id"] for h in hits],
    }

TEST_QUESTIONS = [
    {"q": "Why was the claim flagged for potential fraud?", "expected_categories": ["fraud"]},
    {"q": "What caused the property damage in the basement?", "expected_categories": ["property"]},
    {"q": "Are there any rejected prescription drug or medical reimbursement requests?", "expected_categories": ["health"]},
    {"q": "Why did the premium payment trigger a double deduction?", "expected_categories": ["billing"]}
]

for tq in TEST_QUESTIONS:
    result = ask_qdrant_rag(tq["q"], COLL_COSINE, k=3)
    print(f"── {tq['q']}")
    print(f"   expected categories: {tq['expected_categories']}")
    print(f"   sources retrieved:   {result['sources']}")
    print(f"   answer: {result['answer']}")
    print()

── Why was the claim flagged for potential fraud?
   expected categories: ['fraud']
   sources retrieved:   ['clm_frd_01', 'clm_frd_02', 'clm_hlth_02']
   answer: The claim was flagged for potential fraud because it contained conflicting accident timelines and evidence of pre‑existing body damage. [clm_frd_01]

── What caused the property damage in the basement?
   expected categories: ['property']
   sources retrieved:   ['clm_prop_01', 'clm_prop_02', 'clm_auto_02']
   answer: The damage in the basement was caused by a water pipe burst. [clm_prop_01]

── Are there any rejected prescription drug or medical reimbursement requests?
   expected categories: ['health']
   sources retrieved:   ['clm_hlth_02', 'clm_hlth_01', 'clm_bill_02']
   answer: Yes.  
- The prescription drug claim **clm_hlth_02** was rejected because the prior‑authorization form from the physician was missing. [clm_hlth_02]  
- The hospital stay pre‑authorization request **clm_hlth_01** was denied for out‑of‑network eme

## Cell 23 — Cleanup

In [29]:
for coll in [COLL_HNSW_DEFAULT, COLL_HNSW_TUNED, COLL_QUANT, COLL_COSINE, COLL_DOT, COLL_L2]:
    try:
        qdrant.delete_collection(coll)
        print(f"  deleted {coll}")
    except Exception as e:
        print(f"  skip {coll}: {e}")

print("\nDay 1's claims collection (wk07_day1_claims) is preserved.")

  deleted wk07_day2_hnsw_default
  deleted wk07_day2_hnsw_tuned
  deleted wk07_day2_quantized
  deleted wk07_day2_metric_cosine
  deleted wk07_day2_metric_dot
  deleted wk07_day2_metric_l2

Day 1's claims collection (wk07_day1_claims) is preserved.
